# 07 — Synthesis & Results
**Citi Bike Demand Forecasting & A/B Testing Framework**

This notebook pulls together the key numbers and charts from every phase into a single narrative.
Intended audience: someone who wants the full story without running all six notebooks.

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from src.experiments.ab_test import assign_clusters, simulate_outcomes, run_ttest, run_mannwhitney, run_sequential_test
from src.experiments.bayesian_ab import run_bayesian_ab
from src.experiments.power_analysis import power_curve
from src.models.clustering import CLUSTER_NAMES

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('tab10')

PROCESSED_DIR = Path('../data/processed')

---
## The Problem

Citi Bike's rebalancing problem: bikes pile up at destination stations and drain from origin stations. The current heuristic is demand-blind — trucks move bikes on fixed schedules regardless of actual flow patterns.

**This project answers:** Can ML-driven demand forecasting improve dock availability, and can we design a statistically rigorous experiment to prove it?

---
## Phase 1 — Data

In [ ]:
hourly = pd.read_parquet(PROCESSED_DIR / 'hourly_demand_2024Q1.parquet')
station_summary = pd.read_parquet(PROCESSED_DIR / 'station_summary_2024Q1.parquet')

total_trips    = station_summary['total_trips'].sum()
n_stations     = len(station_summary)
date_range     = f"{hourly['hour'].min().date()} → {hourly['hour'].max().date()}"
top10_share    = station_summary.nlargest(int(n_stations * 0.1), 'total_trips')['total_trips'].sum() / total_trips

print(f"Dataset: {total_trips:,.0f} trips | {n_stations:,} stations | {date_range}")
print(f"Demand concentration: top 10% of stations = {top10_share:.1%} of all trips")

# Hourly demand heatmap (avg trips per station)
pivot = hourly.groupby(['day_of_week', 'hour_of_day'])['trip_count'].mean().unstack()
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(13, 3.5))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0,
            yticklabels=day_labels, cbar_kws={'label': 'Avg trips / station'})
ax.set_title('Avg Station Demand: Hour × Day of Week — Jan–Mar 2024')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

---
## Phase 2 — Station Clustering

K-means (K=4) on per-station demand profiles — chosen over hierarchical clustering for scalability and interpretability. DBSCAN added for geospatial density clustering. Cluster-level structure is what enables valid A/B randomization later.

In [ ]:
clusters = pd.read_parquet(PROCESSED_DIR / 'station_clusters.parquet')
cluster_sizes = clusters['kmeans_cluster'].value_counts().sort_index()
valid_clusters = cluster_sizes[cluster_sizes >= 10].index.tolist()
clusters_valid = clusters[clusters['kmeans_cluster'].isin(valid_clusters)].copy()

print('K-Means clusters (K=4, valid clusters only):')
for cid in sorted(valid_clusters):
    n = cluster_sizes[cid]
    name = CLUSTER_NAMES.get(cid, '?')
    print(f'  Cluster {cid} — {name}: {n:,} stations')

# Geographic scatter
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
fig, ax = plt.subplots(figsize=(7, 8))
for cid in sorted(valid_clusters):
    mask = clusters_valid['kmeans_cluster'] == cid
    ax.scatter(
        clusters_valid.loc[mask, 'lng'],
        clusters_valid.loc[mask, 'lat'],
        s=6, alpha=0.6, label=f'{cid} — {CLUSTER_NAMES.get(cid,"")}',
        color=colors[cid]
    )
ax.set_title('Station Clusters — NYC')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

---
## Phase 3 — Forecasting Model Comparison

Three models compared: Prophet (seasonal baseline, handles holidays out-of-box), SARIMA (classical time series), and XGBoost (lag features + calendar features on tabular hourly data). XGBoost chosen for production because it's faster to train, more interpretable, and competitive on MAPE without requiring a GPU.

In [ ]:
# Re-run a lightweight version of the forecasting comparison for synthesis
from src.models.forecasting import (
    get_station_series, train_test_split, evaluate,
    fit_prophet, predict_prophet,
    make_lag_features, fit_xgboost, predict_xgboost,
)

CUTOFF = '2024-03-01'

station_totals = hourly.groupby('station_id')['trip_count'].sum().rename('total_trips')
clusters_with_demand = clusters_valid.set_index('station_id').join(station_totals)

representatives = (
    clusters_with_demand
    .groupby('kmeans_cluster')
    .apply(lambda g: g['total_trips'].idxmax())
    .rename('station_id')
)

rows = []
for cluster_id, sid in representatives.items():
    series = get_station_series(hourly, sid)
    train, test = train_test_split(series, CUTOFF)
    feats = make_lag_features(series)
    _, test_f = train_test_split(feats, CUTOFF)

    # Prophet
    m = fit_prophet(train)
    metrics_p = evaluate(test['trip_count'].values, predict_prophet(m, test))

    # XGBoost
    _, test_f = train_test_split(feats, CUTOFF)
    train_f, _ = train_test_split(feats, CUTOFF)
    xm = fit_xgboost(train_f)
    metrics_x = evaluate(test_f['trip_count'].values, predict_xgboost(xm, test_f))

    cname = CLUSTER_NAMES.get(cluster_id, str(cluster_id))
    rows += [
        {'Cluster': cname, 'Model': 'Prophet',  **metrics_p},
        {'Cluster': cname, 'Model': 'XGBoost',  **metrics_x},
    ]

comparison = pd.DataFrame(rows)
pivot_mape = comparison.pivot(index='Cluster', columns='Model', values='MAPE')

fig, ax = plt.subplots(figsize=(9, 3.5))
pivot_mape.plot(kind='bar', ax=ax, edgecolor='none', width=0.6)
ax.set_title('MAPE by Model and Cluster — March 2024 Test Set')
ax.set_ylabel('MAPE (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Model')
plt.tight_layout()
plt.show()

print(comparison.to_string(index=False))

---
## Phase 4 — Anomaly Detection

Three methods applied — Z-score (rolling 1-week window), IQR (per station × hour-of-day slot), and Isolation Forest on residuals. Consensus flagging (2+ methods must agree) reduces false positives, producing a cleaner baseline for the A/B experiment.

In [ ]:
anomaly_flags = pd.read_parquet(PROCESSED_DIR / 'anomaly_flags_2024Q1.parquet')

total = len(anomaly_flags)
print('Anomaly detection summary:')
for method in ['zscore_anomaly', 'iqr_anomaly', 'if_anomaly', 'consensus_anomaly']:
    n = anomaly_flags[method].sum()
    label = method.replace('_anomaly', '').replace('_', ' ').title()
    print(f'  {label:20s}: {n:>6,} ({n/total:.2%})')

# Consensus anomalies by day
import matplotlib.dates as mdates
daily = (
    anomaly_flags[anomaly_flags['consensus_anomaly']]
    .groupby(anomaly_flags.loc[anomaly_flags['consensus_anomaly'], 'hour'].dt.date)
    .size()
)
daily.index = pd.to_datetime(daily.index)

fig, ax = plt.subplots(figsize=(13, 3))
ax.bar(daily.index, daily.values, color='coral', width=0.8)
ax.set_title('Consensus Anomalies per Day (flagged by 2+ methods)')
ax.set_ylabel('Flagged station-hours')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.show()

---
## Phase 5 — A/B Test Results

Randomization at the cluster level (not station level) — bikes physically move between stations, so station-level randomization would cause treatment/control interference (SUTVA violations). Outcomes below are **simulated** to demonstrate the full experimental framework; the design is ready to apply to real operational data.

In [ ]:
BASELINE = 0.75
TRUE_LIFT = 0.08

cluster_assignments = assign_clusters(valid_clusters, random_state=42)
outcomes = simulate_outcomes(
    station_clusters=clusters_valid,
    cluster_assignments=cluster_assignments,
    n_days=31,
    baseline_availability=BASELINE,
    true_lift=TRUE_LIFT,
    noise_std=0.05,
    random_state=42,
)

ttest  = run_ttest(outcomes)
mw     = run_mannwhitney(outcomes)
seq    = run_sequential_test(outcomes, n_looks=4, alpha=0.05)
trace, bayes = run_bayesian_ab(outcomes, n_samples=1000, random_seed=42)

uplift_samples = trace.posterior['uplift'].values.flatten()
ctrl_post = trace.posterior['mu_control'].values.flatten()
trt_post  = trace.posterior['mu_treatment'].values.flatten()

In [ ]:
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Distribution by group
ax1 = fig.add_subplot(gs[0, 0])
for group, color in [('control', 'steelblue'), ('treatment', 'coral')]:
    ax1.hist(outcomes[outcomes['group'] == group]['availability_rate'],
             bins=40, alpha=0.6, color=color, label=group, edgecolor='none')
ax1.set_title('Availability Rate Distribution')
ax1.set_xlabel('Availability rate')
ax1.legend(fontsize=8)

# 2. Daily means
ax2 = fig.add_subplot(gs[0, 1])
daily = outcomes.groupby(['day', 'group'])['availability_rate'].mean().unstack()
daily['control'].plot(ax=ax2, color='steelblue', label='Control', linewidth=1.5)
daily['treatment'].plot(ax=ax2, color='coral', label='Treatment', linewidth=1.5)
ax2.set_title('Daily Mean Availability')
ax2.set_xlabel('Day of experiment')
ax2.legend(fontsize=8)

# 3. Sequential test
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(seq['look'], seq['p_value'], marker='o', color='steelblue', label='p-value')
ax3.plot(seq['look'], seq['boundary_alpha'], marker='s', linestyle='--',
         color='red', label="O'B-F boundary")
ax3.set_title("Sequential Test")
ax3.set_xlabel('Interim look')
ax3.set_xticks(seq['look'])
ax3.legend(fontsize=8)

# 4. Posterior uplift
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(uplift_samples, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
ax4.axvline(0, color='black', linewidth=1.5, linestyle='--')
ax4.axvline(bayes['hdi_95_low'],  color='red', linewidth=1, linestyle=':')
ax4.axvline(bayes['hdi_95_high'], color='red', linewidth=1, linestyle=':',
            label=f"95% HDI")
ax4.set_title(f"Posterior Uplift\nP(T>C) = {bayes['prob_treatment_better']:.1%}")
ax4.set_xlabel('Uplift')
ax4.legend(fontsize=8)

# 5. Posterior means
ax5 = fig.add_subplot(gs[1, 1])
ax5.hist(ctrl_post, bins=60, alpha=0.6, color='steelblue', edgecolor='none', label='Control')
ax5.hist(trt_post,  bins=60, alpha=0.6, color='coral',     edgecolor='none', label='Treatment')
ax5.set_title('Posterior Availability Rate')
ax5.set_xlabel('Availability rate')
ax5.legend(fontsize=8)

# 6. Power curve
ax6 = fig.add_subplot(gs[1, 2])
mde_range = np.linspace(0.01, 0.15, 60)
n_per_group = len(outcomes[outcomes['group'] == 'control'])
pows = power_curve(BASELINE, mde_range, n_per_group=n_per_group)
ax6.plot(mde_range * 100, pows, color='steelblue', linewidth=2)
ax6.axhline(0.80, color='gray', linestyle='--', linewidth=1, label='80% power')
ax6.axvline(TRUE_LIFT * 100, color='coral', linestyle='--', linewidth=1,
            label=f'True lift ({TRUE_LIFT:.0%})')
ax6.set_title('Power Curve')
ax6.set_xlabel('MDE (pp)')
ax6.set_ylabel('Power')
ax6.set_ylim(0, 1.05)
ax6.legend(fontsize=8)

plt.suptitle(
    'A/B Test Results — ML Rebalancing vs Heuristic\n'
    '(Simulated outcomes — framework demonstration, ready for real operational data)',
    fontsize=13, y=1.02
)
plt.savefig(PROCESSED_DIR / 'ab_test_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved summary figure → {PROCESSED_DIR}/ab_test_results.png')

In [ ]:
early_stop = seq[seq['reject']]

print('=' * 60)
print('FULL PROJECT RESULTS SUMMARY')
print('=' * 60)
print(f'\n DATA')
print(f'  Trips (Jan–Mar 2024):     {total_trips:>12,.0f}')
print(f'  Active stations:          {n_stations:>12,}')
print(f'  Top 10% station share:    {top10_share:>12.1%}')
print(f'\n CLUSTERING')
for cid in sorted(valid_clusters):
    print(f'  Cluster {cid} ({CLUSTER_NAMES.get(cid,""):15s}): {cluster_sizes[cid]:>5,} stations')
print(f'\n ANOMALY DETECTION')
print(f'  Consensus anomaly rate:   {anomaly_flags["consensus_anomaly"].mean():>12.3%}')
print(f'\n A/B TEST')
print(f'  Simulated true lift:      {TRUE_LIFT:>12.1%}')
print(f'  Observed lift:            {ttest["observed_lift"]:>12.1%}')
print(f'  Welch t-test p-value:     {ttest["p_value"]:>12.4f}  ({"sig" if ttest["significant_at_05"] else "not sig"})')
print(f'  Mann-Whitney p-value:     {mw["p_value"]:>12.4f}  ({"sig" if mw["significant_at_05"] else "not sig"})')
print(f'  Bayesian P(T>C):          {bayes["prob_treatment_better"]:>12.1%}')
print(f'  95% HDI:                  [{bayes["hdi_95_low"]:.4f}, {bayes["hdi_95_high"]:.4f}]')
if len(early_stop):
    print(f'  Sequential early stop:    look {early_stop.iloc[0]["look"]} / day {early_stop.iloc[0]["day_cutoff"]}')
else:
    print(f'  Sequential early stop:    not triggered')
print('=' * 60)

---
## Conclusion

**What we built:**
A complete ML pipeline that goes from raw bike trip data to a production-ready experiment design — including the statistical framework needed to measure whether the intervention actually worked.

**Key takeaways:**

1. **Demand is highly concentrated and predictable.** Top 10% of stations drive the majority of trips. Commuter patterns (dual AM/PM peaks on weekdays) dominate — exactly the signal XGBoost captures with lag features.

2. **XGBoost beats Prophet on MAPE** for most clusters because the 24h and 168h lag features directly encode the periodicity. Prophet wins on interpretability and handles holidays automatically — a realistic operational baseline for comparison.

3. **Cluster-level randomization is non-negotiable.** Station-level A/B would have massive SUTVA violations — bikes physically move between stations. Randomizing at the cluster level preserves experiment integrity.

4. **Bayesian and frequentist analyses agree.** Both detect the simulated 8pp lift clearly. The Bayesian posterior gives operations teams a direct probability statement — more actionable than a p-value when deciding whether to roll out a new policy.

5. **Sequential testing enables early stopping.** O'Brien-Fleming boundaries let us stop the experiment early if the effect is obvious, saving weeks of experiment runtime without inflating Type I error.

6. **Demand decays sharply with distance from Midtown.** Geospatial analysis shows a clear distance-demand relationship: stations within 1km of Midtown account for a disproportionate share of trips. Commuter clusters concentrate in Manhattan; recreational clusters spread outward. This geographic structure informs both rebalancing priority and which clusters to include in the experiment.

---
*All A/B outcomes are simulated to demonstrate the experimental framework. The pipeline is designed to run on real Citi Bike operational data with no changes to the experimental design.*